# Creative-writing generator (laptop **or** Colab)

Custom storytelling SFT via **teacher + LLM-judge**: DeepSeek V4 Flash writes a SHORT, COMPLETE piece from a programmatic prompt (form × genre × theme); a judge call scores it 1-10; only high-scoring, budget-fitting pieces are kept (creative has no gold to verify, so the judge replaces answer-checking). Output: the `creative` SFT source.

`system=""` so the expressive style generalizes to ordinary answers; pieces are length-capped to fit 2048.

Run calibration (`--limit`) first to see yield + a sample, then the full run.

> ⚠️ Colab clones the repo from GitHub — push the latest `sft/` first. Needs `tokenizer_out/tokenizer.json` under `SYNAPSE_DIR` on Drive (for length budgeting).

In [ ]:
# 1. Environment + SYNAPSE_DIR
import os
try:
    from google.colab import drive
    COLAB = True
    drive.mount('/content/drive', force_remount=False)
    SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
except ImportError:
    COLAB = False
    SYNAPSE_DIR = os.environ.get('SYNAPSE_DIR') or os.path.abspath('./synapse')
os.environ['SYNAPSE_DIR'] = SYNAPSE_DIR
print('COLAB =', COLAB, '| SYNAPSE_DIR =', SYNAPSE_DIR)

In [ ]:
# 2. Deps (tokenizers used for length budgeting)
!pip install -q openai datasets sympy python-dotenv tqdm tokenizers

In [ ]:
# 3. Locate the repo (clone on Colab; find it locally on laptop)
import os, subprocess
if COLAB:
    REPO_DIR = '/content/synapse_repo'
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
    else:
        subprocess.run(['git','clone','--depth=1','https://github.com/ajencinas/synapse.git',REPO_DIR], check=True)
else:
    d = os.path.abspath('.')
    while d != os.path.dirname(d) and not os.path.isfile(os.path.join(d,'sft','generate_creative.py')):
        d = os.path.dirname(d)
    REPO_DIR = d
assert os.path.isfile(os.path.join(REPO_DIR,'sft','generate_creative.py')), \
    'generate_creative.py not found — push it to GitHub (Colab) or run from inside the repo (laptop)'
print('REPO_DIR =', REPO_DIR)

In [ ]:
# 4. Teacher key: OPENROUTER_API_KEY *or* DEEPSEEK_API_KEY (auto-detected; DeepSeek preferred).
#    Colab: add in Secrets (🔑). Laptop: read from repo .env automatically.
import os
if COLAB:
    try:
        from google.colab import userdata
        for name in ('OPENROUTER_API_KEY','DEEPSEEK_API_KEY'):
            if not os.environ.get(name):
                v = userdata.get(name)
                if v: os.environ[name] = v
    except Exception:
        pass
print('teacher keys:', [n for n in ('DEEPSEEK_API_KEY','OPENROUTER_API_KEY') if os.environ.get(n)] or 'none — will use repo .env')

In [ ]:
# 5. CALIBRATION — 200 prompts. Prints yield% + reject reasons; inspect a sample below.
#    (judge default keeps only score>=8; lower with --min-score 7 for more volume.)
cmd = f'cd {REPO_DIR} && python sft/generate_creative.py --limit 200 --workers 24'
print(cmd)
!{cmd}

In [ ]:
# 6. FULL RUN — all unique prompt combos (uncapped; add --budget-usd N to hard-cap).
#    Resumable: re-run to continue (no dupes, no re-spend).
cmd = f'cd {REPO_DIR} && python sft/generate_creative.py --workers 24'
print(cmd)
!{cmd}

In [ ]:
# 7. Inspect output
import json, os
base = os.path.join(SYNAPSE_DIR, 'datasets_sft', 'creative')
raw = os.path.join(base, 'creative_raw.jsonl')
n = sum(1 for _ in open(raw)) if os.path.exists(raw) else 0
print('kept pieces:', n)
if n:
    ex = json.loads(open(raw).readline())
    print('score:', ex.get('score'), '| genre:', ex.get('genre'), '| form:', ex.get('form'))
    print('\nPROMPT:', ex['messages'][0]['content'])
    print('\nPIECE:\n' + ex['messages'][1]['content'][:900])
mp = os.path.join(base, 'meta_raw.json')
if os.path.exists(mp):
    print('\nmeta:', json.load(open(mp)))

## After generating → fold into SFT
```bash
python sft/tokenize_sft_data.py --datasets creative --force
python sft/consolidate_sft_data.py
```
Then add `"creative": ~0.06-0.10` to `SFT_DATA_MIX` in `sft.py` **after** tokenizing (the trainer hard-fails on a mix source with no `train.jsonl`).

**Notes**
- No gold to verify — quality is gated by the LLM-judge (`--min-score`, default 8). The teacher judging itself is somewhat lenient; the high threshold + length discipline compensate.
- `system=""` so expressive style generalizes; pieces are capped to fit the 2048 budget.
- Pairs with the `no_robots` OTS source (run via `sft_data_prep.ipynb`) for the hybrid storytelling plan.